# 07 — Pipeline Validation

End-to-end validation of the complete detection pipeline: data loading, preprocessing,
anomaly detection, classification, zero-day identification, and alert generation.
This notebook verifies that every stage works correctly in sequence.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import time
import os
import sys
from datetime import datetime

sys.path.insert(0, os.path.join(os.path.dirname("__file__"), ".."))
from src.features.preprocessor import clean_dataset, encode_labels, scale_features
from src.models.anomaly_detector import IsolationForestDetector
from src.models.classifier import RandomForestClassifier

## 1. Load Raw Data

In [ ]:
raw_path = "../data/raw/Monday-WorkingHours.pcap_ISCX.csv"
if os.path.exists(raw_path):
    df = pd.read_csv(raw_path)
    print(f"Loaded raw data: {df.shape[0]} rows, {df.shape[1]} columns")
else:
    # Fallback: use processed data
    print("Raw CSV not found, loading processed data...")
    df = pd.read_csv("../data/processed/train.csv")
    print(f"Loaded processed data: {df.shape[0]} rows, {df.shape[1]} columns")

print(f"Columns: {list(df.columns[:10])}...")
df.head()

## 2. Full Preprocessing Pipeline

In [ ]:
# Step 1: Clean
df_clean = clean_dataset(df)
print(f"After cleaning: {df_clean.shape}")

# Step 2: Encode labels
label_col = "Label" if "Label" in df_clean.columns else "label"
df_encoded, le = encode_labels(df_clean, label_col)
print(f"Classes: {list(le.classes_)}")

# Step 3: Split features and labels
feature_cols = [c for c in df_encoded.columns if c != label_col]
X = df_encoded[feature_cols].values.astype(np.float32)
y = df_encoded[label_col].values

# Step 4: Scale
X_scaled, scaler = scale_features(X)
print(f"Features scaled: {X_scaled.shape}")

# Step 5: Train/test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

## 3. Anomaly Detection Stage

In [ ]:
# Train or load Isolation Forest
iso_detector = IsolationForestDetector(contamination=0.1, random_state=42)
iso_detector.fit(X_train)

# Predict anomalies
anomaly_preds = iso_detector.predict(X_test)
# -1 = anomaly, 1 = normal
anomaly_labels = np.where(anomaly_preds == -1, 1, 0)  # 1 = flagged as anomaly

n_anomalies = anomaly_labels.sum()
print(f"Total test samples: {len(X_test)}")
print(f"Anomalies detected: {n_anomalies} ({n_anomalies/len(X_test)*100:.1f}%)")
print(f"Normal samples: {len(X_test) - n_anomalies}")

## 4. Classification Stage

In [ ]:
# Train or load Random Forest on training data
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)

# Classify only anomaly-flagged samples
anomaly_mask = anomaly_labels == 1
if anomaly_mask.sum() > 0:
    X_anomalous = X_test[anomaly_mask]
    rf_predictions = rf_clf.predict(X_anomalous)
    rf_proba = rf_clf.predict_proba(X_anomalous)
    print(f"Classified {len(X_anomalous)} anomalous samples")
    print(f"Prediction distribution:")
    unique, counts = np.unique(rf_predictions, return_counts=True)
    for cls, cnt in zip(unique, counts):
        print(f"  {le.inverse_transform([cls])[0]}: {cnt}")
else:
    print("No anomalies detected — nothing to classify")

## 5. Zero-Day Detection Logic

In [ ]:
# Zero-day detection: anomaly=True AND (classified as BENIGN OR low confidence)
CONFIDENCE_THRESHOLD = 0.7
results_df = pd.DataFrame({
    "true_label": y_test,
    "is_anomaly": anomaly_labels.astype(bool),
})

# Initialize RF predictions for all samples
rf_all_preds = np.full(len(X_test), -1)
rf_all_proba = np.zeros((len(X_test), len(le.classes_)))

if anomaly_mask.sum() > 0:
    rf_all_preds[anomaly_mask] = rf_predictions
    rf_all_proba[anomaly_mask] = rf_proba

results_df["rf_prediction"] = rf_all_preds
results_df["rf_confidence"] = rf_all_proba.max(axis=1)

# Zero-day flag: anomaly but RF says BENIGN or low confidence
benign_idx = list(le.classes_).index("BENIGN") if "BENIGN" in le.classes_ else 0
is_zero_day = (
    results_df["is_anomaly"] &
    ((results_df["rf_prediction"] == benign_idx) |
     (results_df["rf_confidence"] < CONFIDENCE_THRESHOLD))
)
results_df["zero_day_flag"] = is_zero_day.values

print(f"Zero-day samples flagged: {is_zero_day.sum()}")
print(f"Sample of zero-day flags:")
results_df[results_df["zero_day_flag"]].head(10)

## 6. Pipeline Statistics

In [ ]:
total = len(X_test)
n_anom = anomaly_mask.sum()
n_classified = anomaly_mask.sum()
n_zero_day = is_zero_day.sum()
n_known_attack = n_anom - n_zero_day

print("=" * 55)
print("PIPELINE STATISTICS")
print("=" * 55)
print(f"  Total samples:            {total}")
print(f"  Anomalies detected:       {n_anom} ({n_anom/total*100:.1f}%)")
print(f"  Classified (RF):          {n_classified}")
print(f"  Known attacks identified: {n_known_attack}")
print(f"  Zero-day flagged:         {n_zero_day}")
print("=" * 55)

# True positive rate for zero-day
true_attacks = (y_test != "BENIGN")
if true_attacks.sum() > 0:
    recall_zd = n_zero_day / true_attacks.sum()
    print(f"  Zero-day recall (vs true attacks): {recall_zd:.2%}")
print("=" * 55)

## 7. Alert Generation

In [ ]:
def generate_alert(idx, row, sample_features=None):
    severity = "CRITICAL" if row["zero_day_flag"] else "HIGH"
    return {
        "timestamp": datetime.now().isoformat(),
        "sample_index": int(idx),
        "severity": severity,
        "anomaly_detected": bool(row["is_anomaly"]),
        "rf_prediction": le.inverse_transform([int(row["rf_prediction"])])[0] if row["rf_prediction"] >= 0 else "N/A",
        "confidence": float(row["rf_confidence"]),
        "zero_day_flag": bool(row["zero_day_flag"]),
        "true_label": row["true_label"],
        "message": f"{'ZERO-DAY' if row['zero_day_flag'] else 'Known attack'} detected at index {idx}"
    }

alerts = []
for idx, row in results_df[results_df["is_anomaly"]].head(50).iterrows():
    alerts.append(generate_alert(idx, row))

alerts_df = pd.DataFrame(alerts)
print(f"Generated {len(alerts_df)} alerts")
print(f"\nSeverity breakdown:")
print(alerts_df["severity"].value_counts().to_string())
print(f"\nFirst 10 alerts:")
alerts_df.head(10)